# 🧪 GPU-проверка: режимы USA Safari — версия 20261030174545

Короткий прогон режимов 1 (Editorial VO), 2 (Stock Ref, openpose) и 5 (Inpaint Anatomy) на малом размере и малом числе шагов. Нужен после обновления кода, чтобы убедиться, что всё запускается на GPU Colab.

Версия в названии — версия этого ноутбука. Первая ячейка печатает версию кода из репозитория: если числа разные, ноутбук устарел, открой свежий. Версия также стоит в имени файла отчёта.

**Запуск только в Colab, на локальном компьютере не запускать.** Включи GPU: Runtime → Change runtime type → T4 (или L4).

**Оценка до запуска (не измерена, пересмотрю по вашему отчёту):**
- Первый запуск: 15–25 минут, почти всё время уходит на скачивание моделей (~11 ГБ, SD 1.5 + 2 ControlNet + inpaint). Модели кешируются на Drive, `MyDrive/hf_cache`.
- Повторный запуск: 2–5 минут.
- Видеопамять: ожидаю пик около 4–6 ГБ при 512 px и 8 шагах, укладывается в T4 (16 ГБ).

**Шаги:** 1) запустить ячейки сверху вниз; 2) прислать мне текст итоговой таблицы из последней ячейки (или JSON из `MyDrive/sd_colab_gallery_reports/`).

Режимы 2 и 5 берут картинку у первого успешного режима; если его нет в списке, проверка рисует простую картинку сама. Свой исходник можно указать в `SOURCE_IMAGE`.

Сначала малый прогон. Если все три режима OK, можно поднять `SIZE` до 768 и `STEPS` до 20.

In [ ]:
#@title 🚀 Репозиторий и Google Drive
import os, sys
REPO_NAME  = "sd-colab-gallery"
ORIGIN_URL = f"https://github.com/tekswirl25/{REPO_NAME}.git"
repo_dir   = f"/content/{REPO_NAME}"
DRIVE_MOUNT  = "/content/drive"                    #@param {type:"string"}
HF_CACHE_DIR = "/content/drive/MyDrive/hf_cache"   #@param {type:"string"}

if not os.path.exists(repo_dir):
    !git clone $ORIGIN_URL $repo_dir
else:
    !git -C $repo_dir pull -q
os.chdir(repo_dir)   # presets/ читаются по относительному пути
if repo_dir not in sys.path:
    sys.path.insert(0, repo_dir)

from google.colab import drive
drive.mount(DRIVE_MOUNT)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE_DIR
from scripts.smoke_check import SMOKE_VERSION
print("🧪 Версия проверки в коде:", SMOKE_VERSION)
print("✅ repo:", repo_dir, "| HF cache:", HF_CACHE_DIR)

In [ ]:
# @title 🧹 Проверка кеша моделей на Drive
# Пустой или недокачанный файл модели в кеше ломает загрузку: IndexError: pop from empty list.
# Ячейка находит такие файлы. DELETE_BROKEN = True удаляет папки затронутых моделей,
# они скачаются заново при следующем запуске (несколько ГБ, пара минут).
DELETE_BROKEN = False  #@param {type:"boolean"}

from scripts import hf_cache_check
found = hf_cache_check.find_broken_files(HF_CACHE_DIR)
broken = hf_cache_check.affected_models(found, HF_CACHE_DIR)
if DELETE_BROKEN and broken:
    hf_cache_check.remove_models(broken, HF_CACHE_DIR)
print(hf_cache_check.format_report(found, HF_CACHE_DIR, deleted=bool(DELETE_BROKEN and broken)))


In [ ]:
# @title 📦 Зависимости
# torch уже стоит в Colab, его не переустанавливаем.
!pip install --quiet diffusers transformers accelerate safetensors opencv-python-headless pillow pyyaml controlnet_aux
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU не включён: Runtime → Change runtime type → T4")
print("✅ GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

In [ ]:
# @title ▶ Проверка режимов
VARIANT    = "SD15"   #@param ["SD15", "SDXL"]
MODES      = "editorial-vo,stock-ref-comic,inpaint-anatomy"  #@param {type:"string"}
STEPS      = 8        #@param {type:"integer"}
SIZE       = 512      #@param {type:"integer"}
SOURCE_IMAGE = ""   #@param {type:"string"}  # необязательно: свой файл-исходник для режимов 2 и 5
OUTPUT_DIR = "/content/outputs_smoke"   #@param {type:"string"}
REPORT_DIR = "/content/drive/MyDrive/sd_colab_gallery_reports"  #@param {type:"string"}

from scripts.config import init_config
from scripts.smoke_check import run_smoke, format_report

CONFIG, _, _, _ = init_config(model_variant=VARIANT, output_dir=OUTPUT_DIR, mode="GPU_OPTIMAL")
report = run_smoke(
    mode_ids=[m.strip() for m in MODES.split(",") if m.strip()],
    config=CONFIG, steps=STEPS, size=SIZE, report_dir=REPORT_DIR,
    source_path=SOURCE_IMAGE or None,
)
print(format_report(report))
print("\nОтчёт сохранён в", REPORT_DIR)

In [ ]:
# @title 🔎 Диагностика весов inpaint (запускать после проверки, ничего не удаляет)
# Показывает, какие файлы весов есть в репозиториях (pickle или безопасные safetensors)
# и что лежит в кеше на месте упавшего файла: размер, zip ли, совпадает ли с Hugging Face.
REPOS = "runwayml/stable-diffusion-inpainting,stable-diffusion-v1-5/stable-diffusion-inpainting"  #@param {type:"string"}
CHECK_CACHED_REPO = "runwayml/stable-diffusion-inpainting"  #@param {type:"string"}

import glob, os
from huggingface_hub import HfApi
from scripts import hf_diagnose

api = HfApi()
remote = {}
for repo in [r.strip() for r in REPOS.split(",") if r.strip()]:
    rows = hf_diagnose.list_weight_files(repo, api)
    print(hf_diagnose.format_weight_table(repo, rows))
    print()
    if repo == CHECK_CACHED_REPO:
        remote = {r["path"]: r["size"] for r in rows if "path" in r}

folder = os.path.join(HF_CACHE_DIR, "models--" + CHECK_CACHED_REPO.replace("/", "--"), "snapshots")
files = sorted(glob.glob(os.path.join(folder, "*", "*", "*.bin")))
print(f"Файлы .bin в кеше {CHECK_CACHED_REPO}: {len(files)}")
for path in files:
    rel = "/".join(path.split(os.sep)[-2:])
    info = hf_diagnose.inspect_local_file(path)
    print(f"  {rel}: размер {info.get('size')}, начало {info.get('head_hex')} -> {hf_diagnose.verdict(info, remote.get(rel))}")
